In [5]:
import os
import json
import urllib.request
import urllib.parse
import requests
import pandas as pd
from pybliometrics.scopus import ScopusSearch
from pybliometrics.utils import create_config, init

# 1. Setup Local Directory
current_dir = os.getcwd()
folder_path = os.path.join(current_dir, 'MuTech', '01_Raw_Exports')
os.makedirs(folder_path, exist_ok=True)
print(f"Ready to save files locally to: {folder_path}\n")

# ==========================================
# SPRINGER NATURE EXTRACTION
# ==========================================
print("Starting Springer extraction...")
springer_api_key = "9f8ca1e2c2b6abe226273578d4045c8a"
springer_query = '("music" OR "instrument") AND ("performance" OR "solo" OR "ensemble") AND ("machine learning" OR "AI") AND year:2020-2025'

# USE PARAMS DICTIONARY FOR AUTOMATIC URL ENCODING
url = "https://api.springernature.com/meta/v2/json"
params = {
    'q': springer_query,
    'api_key': springer_api_key,
    'p': 100
}

try:
    response = requests.get(url, params=params)
    response.raise_for_status() # This will catch HTTP errors
    data = response.json()

    springer_papers = []
    for record in data.get('records', []):
        springer_papers.append({
            'Title': record.get('title'),
            'Abstract': record.get('abstract', 'No abstract available'),
            'Year': record.get('publicationDate', '')[:4],
            'DOI': record.get('doi'),
            'Source': 'Springer'
        })

    df_springer = pd.DataFrame(springer_papers)
    print(f"Total Springer papers extracted: {len(df_springer)}")
    df_springer.to_csv(os.path.join(folder_path, 'Springer_Raw.csv'), index=False)
except requests.exceptions.RequestException as e:
    print(f"Springer extraction failed: {e}")

# ==========================================
# SCOPUS EXTRACTION
# ==========================================
print("\nStarting Scopus extraction...")
# NOTE: If this hangs, run pybliometrics.utils.create_config() manually in a terminal first.
try:
    init() 
except FileNotFoundError:
    print("Scopus config not found. Please run create_config() in your terminal manually.")
    # create_config() # Commented out to prevent freezing in non-interactive environments

try:
    scopus_query = 'TITLE-ABS-KEY(("music" OR "instrument") AND ("performance" OR "solo" OR "ensemble") AND ("machine learning" OR "AI")) AND PUBYEAR > 2019'
    search_results = ScopusSearch(scopus_query)

    if search_results.results:
        df_scopus = pd.DataFrame(search_results.results)
        print(f"Total Scopus papers extracted: {search_results.get_results_size()}")
        df_scopus.to_csv(os.path.join(folder_path, 'Scopus_Raw.csv'), index=False)
    else:
        print("No Scopus results found or query failed.")
except Exception as e:
    print(f"Scopus extraction failed: {e}")

# ==========================================
# IEEE XPLORE EXTRACTION
# ==========================================
print("\nStarting IEEE extraction...")
ieee_api_key = "h24uddpgdmamhmxhmgsfhzg3"
ieee_query = "(music OR instrument) AND (performance OR solo OR ensemble) AND (AI OR machine learning)"
encoded_query = urllib.parse.quote(ieee_query)

# Added start_year and end_year to match your other queries
ieee_url = f"https://ieeexploreapi.ieee.org/api/v1/search/articles?apikey={ieee_api_key}&format=json&max_records=200&start_year=2020&end_year=2025&querytext={encoded_query}"

try:
    response = urllib.request.urlopen(ieee_url)
    data = json.loads(response.read())

    print(f"Total IEEE papers extracted: {data.get('total_records', 0)}")
    if 'articles' in data:
        df_ieee = pd.DataFrame(data['articles'])
        df_ieee.to_csv(os.path.join(folder_path, 'IEEE_Raw.csv'), index=False)
except Exception as e:
    print(f"IEEE extraction failed: {e}")

print("\nAll extractions complete! Check the MuTech/01_Raw_Exports folder.")

Ready to save files locally to: c:\Users\asus\OneDrive\Desktop\Personal\School\DLSU\MuTech\MuTech\01_Raw_Exports

Starting Springer extraction...
Springer extraction failed: 403 Client Error: Forbidden for url: https://api.springernature.com/meta/v2/json?q=%28%22music%22+OR+%22instrument%22%29+AND+%28%22performance%22+OR+%22solo%22+OR+%22ensemble%22%29+AND+%28%22machine+learning%22+OR+%22AI%22%29+AND+year%3A2020-2025&api_key=9f8ca1e2c2b6abe226273578d4045c8a&p=100

Starting Scopus extraction...
Scopus extraction failed: The requestor is not authorized to access the requested view or fields of the resource

Starting IEEE extraction...
IEEE extraction failed: HTTP Error 403: Forbidden

All extractions complete! Check the MuTech/01_Raw_Exports folder.
